In [1]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import BRICS
from rdkit.Chem import rdDepictor
rdDepictor.SetPreferCoordGen(True)
from rdkit.Chem.Draw import IPythonConsole
from rdkit.Chem import Draw
from tqdm import tqdm
import concurrent.futures
import random

In [14]:
df = pd.read_csv("data/linkers.csv", encoding = 'latin1')

In [2]:
head_smarts = [
    # --- Standard Conjugation ---
    "[10*]N1C(=O)C=CC1=O",            # Maleimide
    "[3*]ON1C(=O)CCC1=O",             # NHS Ester
    "[63*]ON1C(=O)CC(S(=O)(=O)O)C1=O",# Sulfo-NHS
    "[4*]CCBr",                       # Bromoethyl
    "[1*]C(=O)CBr",                   # Bromoacetyl
    "[13*][C@@H]1SC[C@@H]2NC(=O)N[C@@H]21", # Biotin
    "[4*]CCC(=O)NN",                  # Hydrazide
    "[16*]c1ccc(C=O)cc1",             # Benzaldehyde
    
    # --- Click Chemistry ---
    "[4*]CCN=[N+]=[N-]",              # Azide
    "[4*]CC#C",                       # Alkyne
    "[5*]N1Cc2ccccc2C#Cc2ccccc21",    # DBCO
    "[15*]C1CC/C=C\\CCC1",            # TCO
    "[14*]c1nnc(C)nn1",               # Tetrazine
    "[15*]C1[C@H]2CCC#CCC[C@@H]12",   # BCN
    "[15*]C1C2CCC#CCCC12",            # Cyclooctyne variant

    # --- Activated Esters / Leaving Groups ---
    "[16*]c1c(F)c(F)c(F)c(F)c1F",     # PFP
    "[16*]c1c(F)c(F)cc(F)c1F",        # TFP
    "[16*]c1ccc([N+](=O)[O-])cc1",    # Nitrophenyl
    "[16*]c1ccc([N+](=O)[O-])cc1[N+](=O)[O-]", # Dinitrophenyl
    "[3*]OS(=O)(=O)c1ccc(C)cc1",      # Tosylate
    "[3*]OS(C)(=O)=O",                # Mesylate
]

tail_smarts = [
    # --- Functional Tails ---
    "[4*]CCC(=O)O",                   # Propionic Acid
    "[4*]CC(=O)O",                    # Acetic Acid
    "[4*]CCN",                        # Ethylamine
    "[5*]NC",                         # Methylamine
    "[16*]c1ccccc1",                  # Phenyl
    "[4*]CCS",                        # Thiol
    "[11*]SC(C)=O",                   # Thioacetate
    "[3*]ON",                         # Hydroxylamine

    # --- Caps & Protecting Groups ---
    "[4*]C(C)(C)C",                   # t-Butyl (Boc)
    "[1*]C(=O)OCC1c2ccccc2-c2ccccc21",# Fmoc (Corrected Carbamate w/ Carbonyl attachment)
    "[4*]CCO",                        # Ether
    "[3*]OC",                         # Methoxy
    "[4*]CC",                         # Ethyl
    "[14*]c1ccccn1",                  # Pyridine
    "[1*]C(N)=O",                     # Amide
    "[4*]CCS(=O)(=O)O",               # Sulfonate
    "[8*]CO",                         # Methoxy variant
    "[1*]C(C)=O",                     # Acetyl
    "[8*]CCC(=O)CC(C)=O",             # Levulinyl
]

cleavable_smarts = [
    # --- Core Linkers ---
    "[16*]c1ccc([16*])cc1",           # PABC Core
    "[11*]SS[11*]",                   # Disulfide Bond
    "[1*]C(=O)NNC(=O)CC[4*]",         # Hydrazone
    "[16*]c1ccc(N=Nc2cc([16*])ccc2O)cc1", # Azo-linker
    "[16*]c1ccc([N+](=O)[O-])c([16*])c1", # Nitro-aromatic

    # --- Peptides ---
    "[1*]C(=O)[C@@H]([4*])C(C)C",     # Valine
    "[1*]C(=O)[C@@H]([4*])C",         # Alanine
    "[4*]CCC[C@H]([4*])C(=O)O",       # Glutamic acid derivative
    "[4*]CCCC[C@H]([4*])C(N)=O",      # Citrulline/Glutamine
    "[1*]C(=O)C([4*])C(C)C",          # Valine derivative
    "[1*]C(=O)CC[C@H]([4*])C(=O)O",   # Glutamic acid
    "[4*]CCCC[C@H]([4*])C(=O)O",      # Lysine
]

spacer_unit_smarts = [
    "[3*]O[3*]", "[4*]CC[4*]", "[5*]N[5*]", 
    "[1*]C(=O)CC[4*]", "[1*]C([1*])=O", "[1*]C(=O)CCCC[8*]", 
    "[4*]C[8*]", "[1*]C(=O)CC[8*]", "[1*]C(=O)NO[3*]", 
    "[1*]C(=O)C[4*]", "[1*]C(=O)CCC([1*])=O", "[4*]CC[8*]", 
    "[1*]C([6*])=O", "[1*]C(=O)CCC[4*]", "[4*]CCC[4*]", 
    "[4*]C([8*])C", "[1*]C(=O)CCCCC[8*]", "[1*]C(=O)C(C[4*])S(=O)(=O)O", 
    "[1*]C(=O)CCCCC([1*])=O", "[5*]N([5*])C", "[4*]CCS(=O)(=O)CC[4*]", 
    "[11*]S[11*]", "[14*]c1noc([14*])n1", "[1*]C(=O)C[8*]", 
    "[1*]C(=O)CCC([4*])C", "[1*]C(=O)CCCCC[4*]", "[4*]CC(N)C[4*]", 
    "[1*]C(=O)CCCCCCCCCCCCCCCCC([1*])=O"
]

In [3]:
def join_fragments(mol_a, mol_b):
    """
    Force-joins two molecules by connecting their first available dummy atoms (*),
    ignoring specific BRICS labels (e.g., connecting [4*] to [4*] is allowed).
    """
    # 1. Combine the two molecules into one disconnected object
    # Chem.CombineMols preserves atom properties but shifts indices for the second mol
    combined = Chem.CombineMols(mol_a, mol_b)
    
    # 2. Convert to an editable molecule (RWMol) to modify bonds/atoms
    rw_mol = Chem.RWMol(combined)
    
    # 3. Find the dummy atoms (atomic number 0)
    # We need to find one dummy from the 'A' side and one from the 'B' side
    # The 'B' side atoms were appended, so they have higher indices
    atoms = rw_mol.GetAtoms()
    dummy_indices = [a.GetIdx() for a in atoms if a.GetAtomicNum() == 0]
    
    if len(dummy_indices) < 2:
        return None # Not enough connection points
    
    # Heuristic: We usually want to connect the "end" of Mol A to the "start" of Mol B.
    # We take the *last* dummy from the list (likely from Mol B) 
    # and the *first* dummy (likely from Mol A) or similar pairing.
    # To be safe and simple: we verify we are connecting two different fragments.
    
    atom_a_idx = dummy_indices[0]
    atom_b_idx = dummy_indices[-1] # Pick the furthest one to avoid self-looping on A
    
    atom_a = rw_mol.GetAtomWithIdx(atom_a_idx)
    atom_b = rw_mol.GetAtomWithIdx(atom_b_idx)
    
    # 4. Identify the neighbors (the real atoms attached to the dummies)
    # Each dummy should have exactly one neighbor
    if atom_a.GetDegree() == 0 or atom_b.GetDegree() == 0:
        return None
        
    neighbor_a_idx = atom_a.GetNeighbors()[0].GetIdx()
    neighbor_b_idx = atom_b.GetNeighbors()[0].GetIdx()
    
    # 5. Add a SINGLE bond between the real atoms
    rw_mol.AddBond(neighbor_a_idx, neighbor_b_idx, Chem.BondType.SINGLE)
    
    # 6. Remove the dummy atoms. 
    # IMPORTANT: Remove higher index first to avoid shifting the lower index!
    # We sort indices in descending order.
    rw_mol.RemoveAtom(max(atom_a_idx, atom_b_idx))
    rw_mol.RemoveAtom(min(atom_a_idx, atom_b_idx))
    
    # 7. Sanitize to ensure the new molecule makes chemical sense
    try:
        Chem.SanitizeMol(rw_mol)
        return rw_mol.GetMol()
    except:
        return None

In [4]:
def build_linker(num_spacers=3):
    # Start with a random Head
    current_mol = Chem.MolFromSmiles(random.choice(head_smarts))
    if current_mol is None: return None

    # Define the sequence of pieces we want to add
    # E.g. Spacer -> Cleavable -> Tail
    structure_sequence = []
    structure_sequence.extend([spacer_unit_smarts] * random.randint(1, 2))
    structure_sequence.append(cleavable_smarts)
    structure_sequence.append(tail_smarts)

    # Build loop
    for candidate_list in structure_sequence:
        candidates = list(candidate_list)
        random.shuffle(candidates)
        
        success = False
        for next_smi in candidates:
            next_frag = Chem.MolFromSmiles(next_smi)
            if next_frag is None: continue

            # This now uses the new PERMISSIVE joiner
            combined_mol = join_fragments(current_mol, next_frag)

            if combined_mol is not None:
                current_mol = combined_mol
                success = True
                break 
        
        if not success: return None

    # Clean up any remaining dummy atoms (*) if possible
    # (Optional: You can just leave them or strip them here)
    try:
        Chem.SanitizeMol(current_mol)
        smi = Chem.MolToSmiles(current_mol)
        # Only accept if it's a decent size
        if current_mol.GetNumAtoms() > 10: 
            return smi
    except:
        return None
    return None

In [7]:
def generate_batch(batch_size):
    """Worker function to generate a batch of unique SMILES."""
    local_batch = set()
    attempts = 0
    max_attempts = batch_size * 50 
    
    while len(local_batch) < batch_size and attempts < max_attempts:
        attempts += 1
        smi = build_linker()
        if smi:
            local_batch.add(smi)
            
    return local_batch

In [9]:
# successes = 0
# attempts = 100

# print(f"Testing {attempts} attempts...")
# for i in range(attempts):
#     mol = build_linker()
#     if mol:
#         successes += 1
#         print(f"Success on attempt {i}!")

# print(f"Total Success Rate: {successes}/{attempts} ({successes/attempts*100}%)")

In [15]:
TARGET_TOTAL = 10_000
BATCH_SIZE = 500
MAX_WORKERS = 8 

# Setup Data containers
# Assuming 'df' is already loaded from your previous cells
existing_mols = set(df['smiles'].unique()) if 'df' in locals() else set()
synthetic_smiles = set()

print(f"Starting Robust Multi-Core Generation: Target {TARGET_TOTAL}")
pbar = tqdm(total=TARGET_TOTAL)

# NOTE: If ProcessPoolExecutor fails in the notebook (pickling errors), 
# replace with 'concurrent.futures.ThreadPoolExecutor'
with concurrent.futures.ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
    # 1. Fill queue
    futures = {executor.submit(generate_batch, BATCH_SIZE) for _ in range(MAX_WORKERS * 2)}
    
    while len(synthetic_smiles) < TARGET_TOTAL:
        # 2. Wait for first result
        done, not_done = concurrent.futures.wait(
            futures, return_when=concurrent.futures.FIRST_COMPLETED
        )
        
        # 3. Process results
        for future in done:
            try:
                batch_results = future.result()
                
                new_mols_count = 0
                for raw_smi in batch_results:
                    if len(synthetic_smiles) >= TARGET_TOTAL: break
                    
                    if raw_smi not in synthetic_smiles:
                        mol = Chem.MolFromSmiles(raw_smi)
                        if mol:
                            canon_smi = Chem.MolToSmiles(mol, canonical=True)
                            if (canon_smi not in synthetic_smiles) and (canon_smi not in existing_mols):
                                synthetic_smiles.add(canon_smi)
                                new_mols_count += 1
                
                pbar.update(new_mols_count)
                futures.remove(future)
                
            except Exception as e:
                print(f"Batch failed: {e}")
                futures.remove(future)

        # 4. Refill queue
        while len(futures) < (MAX_WORKERS * 2) and len(synthetic_smiles) < TARGET_TOTAL:
            futures.add(executor.submit(generate_batch, BATCH_SIZE))
            
pbar.close()
print(f"Finished! Total unique synthetic molecules: {len(synthetic_smiles)}")

Starting Robust Multi-Core Generation: Target 10000


100%|██████████| 10000/10000 [00:02<00:00, 4864.65it/s]

Finished! Total unique synthetic molecules: 10000


In [16]:
new_df = pd.DataFrame()
new_df['smiles'] = list(existing_mols) + list(synthetic_smiles)

In [17]:
new_df.shape

(12768, 1)

In [18]:
new_df.head()

,smiles
0,CC(C)(C)OC(=O)COCCOCCOCCBr
1,CC(C)(C)OC(=O)N[C@H](COCCOCCOCCN=[N+]=[N-])C(=O)O
2,CC(C)(C)OC(=O)CCOCCOCCOCCOCCOCCNC
3,C(COCCOCCON)O
4,C1CC(=O)N(C1=O)OC(=O)CCOCCOCCOCCOCCOCCOCCNC(=O...


In [19]:
new_df.to_pickle("data/synthetic_data.pkl")